# Dealscan Data Cleaning Summary

This notebook processes and cleans the raw Dealscan dataset to prepare it for analysis.  

## Cleaning Steps Overview

### 1. Import Data
- Load the raw Dealscan dataset into a pandas DataFrame.

---

### 2. Currency Filter (Ignored for now)
- **Keep only U.S. Dollar-denominated loans.**
- Use the **`Deal_Currency`** column and retain only rows where the value is **`'US Dollar'`** or **`'U.S. Dollar'`**.
- **DataFrame name after this step:** `df_usd`

---

### 3. Bank Nationality Filter (Ignored for now)
- **Filter out non-U.S. banks** using the **`Lender_Parent_Operating_Country`** column.
- Keep only rows where the value is **`'United States'`**.
- **DataFrame name after this step:** `df_usd_usbanks`

---

### 4. Borrower Industry Dummies
- **Create two dummy variables based on `Major_industry_group`:**
  - **`is_financial_borrower = 1`** if the borrower is in **`'Financial Services'`**
  - **`is_utility_borrower = 1`** if the borrower is in **`'Utilities'`**

---

### 5. Deal Status Filter
- **Exclude deals where `Phase` equals `'In Process'`** 
- **DataFrame name after this step:** `df_usd_usbanks_noProgress`

---

### 6. Zero or Negative Loan Amounts
- **Exclude loans where `Deal_Amount` is less than or equal to 0.**
- **DataFrame name after this step:** `df_usd_usbanks_noProgress_positiveAmount`

---

### 7. Deal Purpose Dummy
- **Create the following dummy variable from `Deal_Purpose`:**
  - **`Deal_purpose_LBO_buyout = 1`** if the purpose is:
    - `'Leveraged Buyout'`
    - `'Sponsored Buyout'`
    - `'Takeover'`
    - `'Acquisition'`
    - `'Merger'`

---

## Final Output
- The cleaned dataset will be saved as: **`2021jan_2024sept_cleaned.csv`**

# STEP 1: Load Data


In [ ]:
import pandas as pd
df = pd.read_csv("2021jan_2024sept.csv")
initial_records = len(df)
print(f"Initial records: {initial_records}")

# STEP 1.5: Explore whether U.S. parent banks participate in non-USD loans


In [ ]:
# First, clean the currency column for consistency
df['Deal_Currency_clean'] = df['Deal_Currency'].str.strip().str.lower()

# Filter rows where the parent bank is based in the United States
df_us_parents = df[df['Lender_Parent_Operating_Country'] == 'United States']

# Summary: total deals by U.S. parent banks
total_us_deals = len(df_us_parents)

# Count how many of these deals are non-USD
usd_labels_clean = ['us dollar', 'u.s. dollar']
us_deals_non_usd = df_us_parents[~df_us_parents['Deal_Currency_clean'].isin(usd_labels_clean)]
num_non_usd_us_banks = len(us_deals_non_usd)

# Calculate percentage
pct_non_usd_us_banks = (num_non_usd_us_banks / total_us_deals) * 100

# Print summary
print(f"Total deals by U.S. parent banks: {total_us_deals}")
print(f"Non-USD deals by U.S. parent banks: {num_non_usd_us_banks}")
print(f"Percentage of U.S. bank deals that are non-USD: {pct_non_usd_us_banks:.2f}%")

# Show top currencies used by U.S. banks
print("\nTop currencies used by U.S. parent banks:")
print(df_us_parents['Deal_Currency_clean'].value_counts())

# STEP 2: Filter to only USD-denominated loans (Ignored For Now)
### keep loans where Deal_Currency is 'US Dollar' or 'U.S. Dollar'

check unique values of currency

In [ ]:
# print("Unique values in Deal_Currency column:")
# print(df['Deal_Currency'].unique())

In [ ]:
# # STEP 1 (Revised): Clean currency column and re-filter for USD loans

# # Standardize the Deal_Currency values: strip spaces and convert to lowercase
# df['Deal_Currency_clean'] = df['Deal_Currency'].str.strip().str.lower()

# # Define acceptable cleaned USD labels
# usd_labels_clean = ['us dollar', 'u.s. dollar']

# # Apply the cleaned filter
# df_usd = df[df['Deal_Currency_clean'].isin(usd_labels_clean)]

# # Recalculate counts
# pre_usd_count = len(df)
# post_usd_count = len(df_usd)
# num_non_usd_eliminated = pre_usd_count - post_usd_count
# pct_non_usd_eliminated = (num_non_usd_eliminated / pre_usd_count) * 100

# print(f"Before USD filter (cleaned): {pre_usd_count} records")
# print(f"After USD filter (cleaned): {post_usd_count} records retained")
# print(f"Non-USD loans eliminated: {num_non_usd_eliminated} ({pct_non_usd_eliminated:.2f}%)")

### Verify Deal_Currency

In [ ]:
# print(df_usd['Deal_Currency'].unique())

# STEP 3: Filter to only U.S.-parented banks (Ignored for Now)

check unique values of Lender_parent_operating_country 

In [ ]:
# print("Unique values in Lender_Parent_Operating_Country column:")
# print(df['Lender_Parent_Operating_Country'].unique())

In [ ]:
# # Keep only rows where Lender_Parent_Operating_Country is exactly 'United States'
# pre_bank = len(df)
# df_usd_usbanks = df[df['Lender_Parent_Operating_Country'] == 'United States']
# post_bank = len(df_usd_usbanks)

# # Print filter summary
# print(f"Before filtering for U.S. parent banks: {pre_bank} records")
# print(f"After filtering for U.S. parent banks: {post_bank} records retained")
# print(f"Non-U.S. bank records eliminated: {pre_bank - post_bank} ({(pre_bank - post_bank) / pre_bank:.2%})")

### Verify Lender_Parent_Operating_Country

In [ ]:
print(df['Lender_Parent_Operating_Country'].unique())

# STEP 4: Create Borrower Industry Dummies

check Major_industry_group unique values

In [ ]:
df['Major_Industry_Group'].unique()

In [ ]:
# Create borrower industry dummies
# is_financial_borrower = 1 if 'Financial Services'
# is_utility_borrower = 1 if 'Utilities'

df.loc[:, 'is_financial_borrower'] = (
    df['Major_Industry_Group'] == 'Financial Services'
).astype(int)

df.loc[:, 'is_utility_borrower'] = (
    df['Major_Industry_Group'] == 'Utilities'
).astype(int)


### Verify Dummies

In [ ]:
df[['is_financial_borrower', 'is_utility_borrower', 'Major_Industry_Group']]

# STEP 5: Exclude in-progress deals

check Phase variable unique values

In [ ]:
df['Phase'].unique()

In [ ]:
pre_phase = len(df)

# Drop rows where 'Phase' is 'In Process' (case-insensitive match)
df_usd_usbanks_noProgress = df[df['Phase'].str.lower() != 'in process']

post_phase = len(df_usd_usbanks_noProgress)

# Print summary
print(f"Before filtering in-process deals: {pre_phase} records")
print(f"After excluding in-process deals: {post_phase} records retained")
print(f"In-process deals excluded: {pre_phase - post_phase} ({(pre_phase - post_phase) / pre_phase:.2%})")

In [ ]:
df_usd_usbanks_noProgress['Phase'].unique()

# STEP 6: Exclude loans with zero or negative Deal_Amount


In [ ]:
pre_amount = len(df_usd_usbanks_noProgress)

# Keep only rows where Deal_Amount is greater than 0
df_usd_usbanks_noProgress_positiveAmount = df_usd_usbanks_noProgress[df_usd_usbanks_noProgress['Deal_Amount'] > 0]

post_amount = len(df_usd_usbanks_noProgress_positiveAmount)

# Print summary
print(f"Before filtering non-positive loan amounts: {pre_amount} records")
print(f"After excluding zero or negative Deal_Amount: {post_amount} records retained")
print(f"Records eliminated: {pre_amount - post_amount} ({(pre_amount - post_amount) / pre_amount:.2%})")

### Verify Deal Amount has positive values

In [ ]:
df_usd_usbanks_noProgress_positiveAmount[df_usd_usbanks_noProgress_positiveAmount['Deal_Amount'] < 0]

# STEP 7: Create dummy for LBO/buyout loan purposes

check Deal_Purpose variable unique values

In [ ]:
df_usd_usbanks_noProgress_positiveAmount['Deal_Purpose'].unique()

In [ ]:
lbo_keywords = ['Leveraged Buyout', 'Sponsored Buyout', 'Takeover', 'Acquisition', 'Merger']

df_usd_usbanks_noProgress_positiveAmount.loc[:, 'Deal_purpose_LBO_buyout'] = (
    df_usd_usbanks_noProgress_positiveAmount['Deal_Purpose']
    .isin(lbo_keywords)
    .astype(int)
)

# Count and percentage
num_lbo = df_usd_usbanks_noProgress_positiveAmount['Deal_purpose_LBO_buyout'].sum()
total_deals = len(df_usd_usbanks_noProgress_positiveAmount)
percentage_lbo = (num_lbo / total_deals) * 100

print(f"Number of deals flagged as LBO or buyout: {num_lbo} ({percentage_lbo:.2f}%)")

Verify Deal Purpose Filtering

In [ ]:
df_usd_usbanks_noProgress_positiveAmount[['Deal_Purpose',"Deal_purpose_LBO_buyout"]]

# Final Output

In [ ]:
# 📊 Summary of Final Cleaned DataFrame

# Total initial records before any filtering
print(f"Initial number of records before cleaning: {initial_records}")

# Final record count
final_records = df_usd_usbanks_noProgress_positiveAmount.shape[0]
removed_records = initial_records - final_records
removed_percent = (removed_records / initial_records) * 100

# Print dataset shape summary
print(f"✅ Total records in cleaned dataset: {final_records}")
print(f"📐 Total columns: {df_usd_usbanks_noProgress_positiveAmount.shape[1]}")
print(f"❌ Records removed during cleaning: {removed_records} ({removed_percent:.2f}%)")
print("\n")

# ✨ Highlight: Columns added or changed during cleaning
highlight_columns = [
    'is_financial_borrower',         # Created from Major_industry_group
    'is_utility_borrower',           # Created from Major_industry_group
    'Deal_purpose_LBO_buyout'        # Created from Deal_Purpose
]

print("\n✨ Highlight: Custom or modified columns in this dataset:")
for col in highlight_columns:
    if col in df_usd_usbanks_noProgress_positiveAmount.columns:
        print(f" - {col}:")
        print(f"   Unique values: {df_usd_usbanks_noProgress_positiveAmount[col].unique()}")
        print(f"   Value counts:\n{df_usd_usbanks_noProgress_positiveAmount[col].value_counts()}\n")
    else:
        print(f" - {col}: ❌ Not found in DataFrame")

### Make sure Step2 filtering currency and Step3 filtering US banks are ignored

In [ ]:
print(df_usd_usbanks_noProgress_positiveAmount['Lender_Parent_Operating_Country'].unique())

In [ ]:
print(df_usd_usbanks_noProgress_positiveAmount['Deal_Currency'].unique())

In [ ]:
df_usd_usbanks_noProgress_positiveAmount

In [ ]:
df_usd_usbanks_noProgress_positiveAmount.to_csv("2021jan_2024sept_cleaned.csv", index=False)